Write a Python decorator called @retry(max_attempts=3, delay=1)
that automatically retries a function if it raises an exception,
with exponential backoff.

In [1]:
import time
import functools
import logging

logging.basicConfig(level=logging.INFO)


def retry(max_attempts=3, delay=1):
    """
    Decorator that retries a function if it raises an exception.

    Args:
        max_attempts (int): maximum retry attempts
        delay (int/float): initial delay between retries (seconds)

    Uses exponential backoff:
        delay * 2^(attempt-1)
    """

    def decorator(func):

        @functools.wraps(func)
        def wrapper(*args, **kwargs):

            attempt = 1
            current_delay = delay

            while attempt <= max_attempts:
                try:
                    return func(*args, **kwargs)

                except Exception as e:
                    if attempt == max_attempts:
                        logging.error(
                            f"{func.__name__} failed after {max_attempts} attempts: {e}"
                        )
                        raise

                    logging.warning(
                        f"Attempt {attempt} failed for {func.__name__}: {e}. "
                        f"Retrying in {current_delay} seconds..."
                    )

                    time.sleep(current_delay)

                    # exponential backoff
                    current_delay *= 2
                    attempt += 1

        return wrapper

    return decorator

In [2]:
@retry(max_attempts=4, delay=1)
def unstable_task():
    import random

    if random.random() < 0.7:
        raise ValueError("Random failure")

    return "Success"


print(unstable_task())

Success
